In [1]:
!pip install -r 'requirements.txt'

Defaulting to user installation because normal site-packages is not writeable


In [1]:

from huggingface_hub import login
import os
import gc 
import torch
from transformers import AutoModelForCausalLM,AutoTokenizer,BitsAndBytesConfig,TrainingArguments
from peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training,PeftModel,TaskType
from datasets import load_dataset
from trl import SFTTrainer
from dotenv import load_dotenv




/home/abhin/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
token=os.getenv('hf_token')
login(token)

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to /home/abhin/.cache/huggingface/token
Login successful


In [3]:
model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
dataset_path='dataset/extended_dataset.jsonl'
output_dir='llama-output'
adapter_save_path='temp-adapters'
merged_save_path='llama-8b-merged-ultra'


In [4]:
GPU_VRAM       = "5.5GiB"   
CPU_RAM        = "7.2GiB"

In [5]:
MAX_SEQ_LEN    = 512        
LORA_R         = 16         
LORA_ALPHA     = 32         
EPOCHS         = 5
LEARNING_RATE  = 2e-4

In [9]:
bnb_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
                              bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)

In [10]:
print("📥 Loading model with GPU + CPU memory split...")
max_memory = {
    0: GPU_VRAM,     
    "cpu": CPU_RAM,   
}

📥 Loading model with GPU + CPU memory split...


In [9]:
print("📥 Loading Llama-3-8B base model...")
model=AutoModelForCausalLM.from_pretrained(model_name,max_memory=max_memory,quantization_config=bnb_config,device_map='auto',trust_remote_code=True,low_cpu_mem_usage=True,torch_dtype=torch.float16)

📥 Loading Llama-3-8B base model...


/home/abhin/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 4/4 [00:15<00:00,  3.82s/it]


In [10]:
tokenizer=AutoTokenizer.from_pretrained(model_name,trust_remote_code=True)
tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side='right'
tokenizer.model_max_length = MAX_SEQ_LEN

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [11]:
print("\n📍 Layer device map:")
device_counts = {}
for name, param in model.named_parameters():
    device = str(param.device)
    device_counts[device] = device_counts.get(device, 0) + 1
for device, count in device_counts.items():
    print(f"  {device}: {count} parameter tensors")
print()


📍 Layer device map:
  cuda:0: 291 parameter tensors



In [12]:
print(f"loading the dataset from {dataset_path}")
raw_dataset=load_dataset('json',data_files=dataset_path,split='train')
dataset=raw_dataset.train_test_split(test_size=0.2,seed=42)



loading the dataset from dataset/extended_dataset.jsonl


In [13]:
def formatting_func(example):
    """Alpaca-style prompt format — handles both single and batched examples"""
    
    # Handle batched input (list of examples)
    if isinstance(example["instruction"], list):
        outputs = []
        for instruction, inp, output in zip(
            example["instruction"],
            example["input"],
            example["output"]
        ):
            if inp and inp.strip():
                text = (
                    f"### Instruction:\n{instruction}\n\n"
                    f"### Input:\n{inp}\n\n"
                    f"### Output:\n{output}"
                )
            else:
                text = (
                    f"### Instruction:\n{instruction}\n\n"
                    f"### Output:\n{output}"
                )
            outputs.append(text)
        return outputs

    # Handle single example
    instruction = example.get("instruction", "")
    inp         = example.get("input", "")
    output      = example.get("output", "")

    if inp and inp.strip():
        return (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{inp}\n\n"
            f"### Output:\n{output}"
        )
    else:
        return (
            f"### Instruction:\n{instruction}\n\n"
            f"### Output:\n{output}"
        )

In [14]:
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

In [ ]:

; # ============================================================
; # STEP 7 — TRAINING ARGUMENTS
; # ============================================================
; training_args = TrainingArguments(
;     output_dir=output_dir,

;     # Batch settings — safe for 6GB
;     per_device_train_batch_size=1,
;     gradient_accumulation_steps=8,       # Effective batch size = 8
;     per_device_eval_batch_size=1,

;     # Training length
;     num_train_epochs=EPOCHS,
;     warmup_steps=10,
;     learning_rate=LEARNING_RATE,
;     max_grad_norm=0.3,                   # Gradient clipping for stability

;     # Precision
;     fp16=True,
;     bf16=False,                          # BF16 only on Ampere+ (RTX 30xx/40xx)

;     # Memory savers
;     gradient_checkpointing=True,         # Trades compute for memory
;     optim="paged_adamw_8bit",            # 8-bit optimizer saves ~400MB vs 32-bit
;     dataloader_num_workers=0,            # Avoids multiprocessing memory issues

;     # Logging & saving
;     logging_steps=5,
;     eval_strategy="steps",
;     eval_steps=50,
;     save_strategy="steps",
;     save_steps=50,
;     save_total_limit=2,                  # Keep only best 2 checkpoints
;     load_best_model_at_end=True,
;     report_to="none",
; )

; # ============================================================
; # STEP 8 — TRAIN
; # ============================================================
; trainer = SFTTrainer(
;     model=model,
;     train_dataset=dataset["train"],
;     eval_dataset=dataset["test"],
;     formatting_func=formatting_func,
;     max_seq_length=MAX_SEQ_LEN,
;     args=training_args,
; )

; # Show VRAM before training starts
; if torch.cuda.is_available():
;     allocated = torch.cuda.memory_allocated(0) / 1024**3
;     reserved  = torch.cuda.memory_reserved(0) / 1024**3
;     print(f"💾 VRAM before training: {allocated:.2f}GB allocated / {reserved:.2f}GB reserved")

; print("🚀 Starting QLoRA fine-tuning...\n")
; trainer.train()

; print("\n💾 Saving adapters...")
; trainer.model.save_pretrained(adapter_save)
; tokenizer.save_pretrained(adapter_save)
; print(f"  Adapters saved to: {adapter_save}")

; # ============================================================
; # STEP 9 — MERGE ADAPTERS INTO BASE MODEL (ON CPU)
; # ============================================================
; print("\n🧹 Clearing VRAM for merge step...")
; del model
; del trainer
; gc.collect()
; torch.cuda.empty_cache()

; if torch.cuda.is_available():
;     free = (torch.cuda.get_device_properties(0).total_memory
;             - torch.cuda.memory_allocated(0)) / 1024**3
;     print(f"  VRAM freed. Available: {free:.2f}GB")

; print("\n🔄 Loading base model on CPU for merging...")
; print("  (This uses ~16GB system RAM — may take a few minutes)")

; base_model = AutoModelForCausalLM.from_pretrained(
;     model_name,
;     low_cpu_mem_usage=True,
;     torch_dtype=torch.float16,
;     device_map="cpu",              # Always merge on CPU to avoid OOM
; )

; print("🔗 Applying LoRA adapters...")
; model_to_merge = PeftModel.from_pretrained(base_model, adapter_save)

; print("⚙️  Merging weights (this takes a few minutes)...")
; merged_model = model_to_merge.merge_and_unload()

; print(f"✅ Saving merged model to: {merged_save}")
; merged_model.save_pretrained(merged_save, safe_serialization=True)
; tokenizer.save_pretrained(merged_save)

; # Clean up CPU RAM
; del base_model, model_to_merge, merged_model
; gc.collect()
; print("  Merged model saved successfully.\n")

; # ============================================================
; # STEP 10 — CONVERT TO GGUF (llama.cpp)
; # ============================================================
; def run(cmd, desc=""):
;     """Run a shell command and stream output"""
;     if desc:
;         print(f"  {desc}")
;     result = subprocess.run(cmd, shell=True, text=True)
;     if result.returncode != 0:
;         print(f"  ⚠️  Command failed: {cmd}")
;         return False
;     return True

; print("📥 Setting up llama.cpp...")

; if not os.path.exists("llama.cpp"):
;     run("git clone https://github.com/ggerganov/llama.cpp", "Cloning llama.cpp...")

; run("pip install -r llama.cpp/requirements.txt -q", "Installing llama.cpp requirements...")

; # Build llama.cpp (Linux/Mac)
; if sys.platform != "win32":
;     run("cd llama.cpp && make -j4", "Building llama.cpp...")
; else:
;     # Windows — use cmake
;     run(
;         "cd llama.cpp && cmake -B build && cmake --build build --config Release",
;         "Building llama.cpp (Windows)..."
;     )

; print("\n🔄 Converting to FP16 GGUF...")
; run(
;     f"python llama.cpp/convert_hf_to_gguf.py {merged_save} "
;     f"--outfile {fp16_gguf} --outtype f16",
;     f"Output: {fp16_gguf}"
; )

; print("\n🗜️  Quantising to Q4_K_M...")
; quantize_bin = "./llama.cpp/llama-quantize" if sys.platform != "win32" \
;                else ".\\llama.cpp\\build\\bin\\Release\\llama-quantize.exe"
; run(
;     f"{quantize_bin} {fp16_gguf} {final_gguf} Q4_K_M",
;     f"Output: {final_gguf}"
; )

; # ============================================================
; # STEP 11 — CLEANUP INTERMEDIATE FILES
; # ============================================================
; if os.path.exists(final_gguf):
;     print("\n🧹 Cleaning up intermediate files...")
;     if os.path.exists(fp16_gguf):
;         os.remove(fp16_gguf)
;         print(f"  Deleted: {fp16_gguf}")
;     if os.path.exists(merged_save):
;         import shutil
;         shutil.rmtree(merged_save)
;         print(f"  Deleted: {merged_save}/")
;     if os.path.exists(adapter_save):
;         import shutil
;         shutil.rmtree(adapter_save)
;         print(f"  Deleted: {adapter_save}/")

;     final_size = os.path.getsize(final_gguf) / 1024**3
;     print(f"\n🎉 Done! Final model: {final_gguf} ({final_size:.1f} GB)")
;     print("\n📋 To run with llama.cpp:")
;     print(f"  ./llama.cpp/llama-cli -m {final_gguf} -p 'Your prompt here' -n 200")
;     print("\n📋 To run with Ollama:")
;     print(f"  ollama create my-llama -f Modelfile  # (point Modelfile to {final_gguf})")
; else:
;     print("\n⚠️  GGUF conversion may have failed.")
;     print(f"  Check if {fp16_gguf} exists and re-run the quantize step manually.")
;     print(f"  Merged HF model is still available at: {merged_save}")

; # ============================================================
; ; # MEMORY TIPS (printed at end for reference)?
; # ============================================================
; print("""
; ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
; 💡 If you hit OOM errors, try these in order:
;    1. Lower MAX_SEQ_LEN from 512 → 256
;    2. Lower LORA_R from 16 → 8
;    3. Lower CPU_RAM if system RAM is limited
;    4. Remove "k_proj"/"o_proj" from target_modules
;    5. Set GPU_VRAM to "5GiB" (more conservative)
; ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
; """)

In [15]:
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    # q_proj + v_proj = minimum set for good results on 6GB
    # Add "k_proj", "o_proj" if you have headroom
    target_modules=["q_proj", "v_proj"],
    bias="none",
)


In [16]:
model = get_peft_model(model, peft_config)

print("📐 Trainable parameters:")
model.print_trainable_parameters()
print()

📐 Trainable parameters:
trainable params: 6,815,744 || all params: 8,037,076,992 || trainable%: 0.08480376642881861



In [18]:
training_args = TrainingArguments(
    output_dir=output_dir,

    # Batch settings — safe for 6GB
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,       # Effective batch size = 8
    per_device_eval_batch_size=1,

    # Training length
    num_train_epochs=EPOCHS,
    warmup_steps=10,
    learning_rate=LEARNING_RATE,
    max_grad_norm=0.3,                   # Gradient clipping for stability

    # Precision
    fp16=True,
    bf16=False,                          # BF16 only on Ampere+ (RTX 30xx/40xx)

    # Memory savers
    gradient_checkpointing=True,         # Trades compute for memory
    optim="paged_adamw_8bit",            # 8-bit optimizer saves ~400MB vs 32-bit
    dataloader_num_workers=0,            # Avoids multiprocessing memory issues

    # Logging & saving
    logging_steps=5,
    evaluation_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,                  # Keep only best 2 checkpoints
    load_best_model_at_end=True,
    report_to="none",
)


In [20]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    formatting_func=formatting_func,
    max_seq_length=MAX_SEQ_LEN,
    args=training_args,
)

/home/abhin/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Map: 100%|██████████| 95/95 [00:00<00:00, 2741.71 examples/s]


In [21]:
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved  = torch.cuda.memory_reserved(0) / 1024**3
    print(f"💾 VRAM before training: {allocated:.2f}GB allocated / {reserved:.2f}GB reserved")

print("🚀 Starting QLoRA fine-tuning...\n")
trainer.train()

print("\n💾 Saving adapters...")
trainer.model.save_pretrained(adapter_save_path)
tokenizer.save_pretrained(adapter_save_path)
print(f"  Adapters saved to: {adapter_save_path}")

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


💾 VRAM before training: 7.42GB allocated / 9.40GB reserved
🚀 Starting QLoRA fine-tuning...



/home/abhin/.local/lib/python3.10/site-packages/torch/utils/checkpoint.py:460: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss,Validation Loss
50,0.609900,0.530588
100,0.252300,0.289043
150,0.152800,0.257993
200,0.125900,0.252020


/home/abhin/.local/lib/python3.10/site-packages/torch/utils/checkpoint.py:460: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/home/abhin/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/abhin/.local/lib/python3.10/site-packages/torch/utils/checkpoint.py:460: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be Fal


💾 Saving adapters...


/home/abhin/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  Adapters saved to: temp-adapters


In [22]:
print("\n🧹 Clearing VRAM for merge step...")
del model
del trainer
gc.collect()
torch.cuda.empty_cache()


🧹 Clearing VRAM for merge step...


In [3]:
if torch.cuda.is_available():
    free = (torch.cuda.get_device_properties(0).total_memory
            - torch.cuda.memory_allocated(0)) / 1024**3
    print(f"  VRAM freed. Available: {free:.2f}GB")

print("\n🔄 Loading base model on CPU for merging...")
print("  (This uses ~16GB system RAM — may take a few minutes)")


  VRAM freed. Available: 6.00GB

🔄 Loading base model on CPU for merging...
  (This uses ~16GB system RAM — may take a few minutes)


In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,max_memory=max_memory,
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,
    device_map="auto",              # Always merge on CPU to avoid OOM
)

print("🔗 Applying LoRA adapters...")
model_to_merge = PeftModel.from_pretrained(base_model, adapter_save_path)

print("⚙️  Merging weights (this takes a few minutes)...")
merged_model = model_to_merge.merge_and_unload()

print(f"✅ Saving merged model to: {merged_save_path}")
merged_model.save_pretrained(merged_save_path, safe_serialization=True)
tokenizer.save_pretrained(merged_save_path)

# Clean up CPU RAM
del base_model, model_to_merge, merged_model
gc.collect()
print("  Merged model saved successfully.\n")

/home/abhin/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards:  25%|██▌       | 1/4 [00:06<00:20,  6.73s/it]

: 

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
